# Module NLP (Rapport_Collecte)

Objectif: extraire des caracteristiques a partir de la colonne textuelle et comparer plusieurs vectorisations + classifieurs.

Dataset attendu: `backend/dataset_ProjetML_2026.csv`.


In [5]:
from pathlib import Path
import re

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, f1_score
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC

DATA_PATH = Path("..") / "backend" / "dataset_ProjetML_2026.csv"
RANDOM_STATE = 42

df = pd.read_csv(DATA_PATH)

# Garder uniquement les lignes avec texte et label
text_col = "Rapport_Collecte"
label_col = "Categorie"

df = df.dropna(subset=[text_col, label_col]).copy()
print("Rows:", len(df))


Rows: 9986


## 1. Nettoyage de texte

On applique un nettoyage simple (lowercase, suppression ponctuation, espaces).

In [6]:
def clean_text(text: str) -> str:
    text = text.lower()
    text = re.sub(r"[^a-zA-Zàâçéèêëîïôûùüÿñæœ0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


df[text_col] = df[text_col].astype(str).map(clean_text)

X_train, X_test, y_train, y_test = train_test_split(
    df[text_col],
    df[label_col],
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=df[label_col],
)


## 2. Vectorisation + classification

Comparaison BoW, TF-IDF et differents classifieurs.

In [7]:
pipelines = {
    "bow_nb": Pipeline([
        ("vec", CountVectorizer(min_df=2)),
        ("clf", MultinomialNB()),
    ]),
    "tfidf_lr": Pipeline([
        ("vec", TfidfVectorizer(ngram_range=(1, 2), min_df=2)),
        ("clf", LogisticRegression(max_iter=2000)),
    ]),
    "tfidf_svc": Pipeline([
        ("vec", TfidfVectorizer(ngram_range=(1, 2), min_df=2)),
        ("clf", LinearSVC()),
    ]),
}

for name, pipe in pipelines.items():
    pipe.fit(X_train, y_train)
    preds = pipe.predict(X_test)
    acc = accuracy_score(y_test, preds)
    f1 = f1_score(y_test, preds, average="weighted")
    print(name, "acc=", round(acc, 4), "f1=", round(f1, 4))


bow_nb acc= 1.0 f1= 1.0
tfidf_lr acc= 1.0 f1= 1.0
tfidf_svc acc= 1.0 f1= 1.0


C:\Users\DELL\AppData\Roaming\Python\Python312\site-packages\sklearn\svm\_classes.py:31: FutureWarning: The default value of `dual` will change from `True` to `'auto'` in 1.5. Set the value of `dual` explicitly to suppress the warning.
  warnings.warn(


## 3. Optionnel: Word2Vec / FastText (Gensim)

Si `gensim` est installe, on peut tester des embeddings.

In [8]:
try:
    from gensim.models import Word2Vec, FastText

    print("Gensim disponible")
except Exception as exc:
    print("Gensim non disponible:", exc)


Gensim non disponible: cannot import name 'triu' from 'scipy.linalg' (C:\Users\DELL\AppData\Roaming\Python\Python312\site-packages\scipy\linalg\__init__.py)
